# Qwen3.5-4B Telecom RCA: GRPO Reinforcement Learning

This notebook runs GRPO (Group Relative Policy Optimization) on top of the SFT-fine-tuned `unsloth/Qwen3.5-4B` model for telecom root-cause analysis. It loads the SFT LoRA adapter produced by `unsloth_sft.ipynb`, then uses vLLM-based fast generation and the TRL `GRPOTrainer` to optimise the model's reasoning policy.

Key rules:

- The SFT adapter preserves the native `<think>...</think>\\boxed{C#}` format. No custom tags are introduced.
- GRPO data (`data/train.json`, 2,400 questions) contains only prompts + target labels; no reasoning traces are supplied to the optimiser.
- Reward functions score format adherence and answer correctness.
- Output is a LoRA adapter that stacks on top of the SFT adapter.


## 1. Installation

Uses vLLM for fast generation (matching the Unsloth GRPO reference). The first run compiles kernels and can take several minutes.

Use a GPU runtime: **Runtime → Change runtime type → T4 GPU**.

In [10]:
!git clone https://github.com/joshsalako/telelogs.git

Cloning into 'telelogs'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 195 (delta 0), reused 1 (delta 0), pack-reused 192 (from 1)
Receiving objects: 100% (195/195), 82.76 MiB | 27.86 MiB/s, done.
Resolving deltas: 100% (77/77), done.


In [3]:
import os
import sys
import importlib.util

print("Python:", sys.executable)

# Install uv with pip, not with uv itself.
!{sys.executable} -m pip install -q --upgrade uv

# Tell uv to install into the notebook's Python environment.
os.environ["UV_SYSTEM_PYTHON"] = "1"

Python: /usr/bin/python3


In [3]:
import subprocess

try:
    import numpy
    import PIL

    _numpy = f"numpy=={numpy.__version__}"
    _pillow = f"pillow=={PIL.__version__}"
except Exception:
    _numpy = "numpy"
    _pillow = "pillow"

try:
    gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
    ).strip()

    is_t4 = "T4" in gpu_name
except Exception:
    gpu_name = "Unknown"
    is_t4 = False

print("GPU:", gpu_name)
print("T4 detected:", is_t4)

GPU: Tesla T4
T4 detected: True


In [1]:
!rm -rf ~/unsloth_compiled_cache
!rm -rf /root/.cache/unsloth_compiled_cache

In [ ]:
!uv pip install --system  \
    {_numpy} \
    {_pillow} \
    torchvision \
    bitsandbytes \
    xformers \
    datasets \
    pandas

!uv pip install --system  \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo.git" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth.git"

!uv pip install vllm==0.15.1

!uv pip install --upgrade "transformers>=5.2.0"

error: Failed to parse: `{_numpy}`
  Caused by: Expected package name starting with an alphanumeric character, found `{`
    {_numpy}
    ^
Using Python 3.12.13 environment at: /usr
Resolved 101 packages in 436ms                                       
Checked 101 packages in 1ms
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 110.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 58.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 

In [1]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch",
    "transformers",
    "trl",
    "unsloth",
    "unsloth_zoo",
    "tokenizers",
    "bitsandbytes",
    "xformers",
    "torchao",
    "vllm",
]

for package in packages:
    try:
        print(f"{package:25s}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:25s}: NOT INSTALLED")

torch                    : 2.9.1
transformers             : 5.14.1
trl                      : 0.24.0
unsloth                  : 2026.7.5
unsloth_zoo              : 2026.7.6
tokenizers               : 0.22.2
bitsandbytes             : 0.50.0
xformers                 : 0.0.33.post2
torchao                  : 0.17.0
vllm                     : 0.15.1


## 2. Configuration

Upload `data/train.json` and the SFT adapter directory (`qwen35_4b_sft_lora/`) to the working directory before running. The notebook also checks `/content/drive/MyDrive/`.

In [4]:
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset

SEED = 42
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 8192
LORA_RANK = 16
SFT_ADAPTER_DIR = "qwen35_4b_sft_lora"
OUTPUT_DIR = "qwen35_4b_grpo_outputs"
GRPO_ADAPTER_DIR = "qwen35_4b_grpo_lora"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "A CUDA GPU runtime is required."
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

SYSTEM_PROMPT = (
    "You are a senior telecom root-cause analysis engineer. Analyze the supplied "
    "drive-test and engineering evidence carefully. Follow the candidate identifiers "
    "defined in the user prompt and finish with exactly one selected identifier "
    "enclosed in \\boxed{}."
)


def locate_path(name):
    candidates = [
        Path(name),
        Path("/content/telelogs") / name,
        Path("/root/telelogs") / name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {name}. Upload it to Colab or place it in MyDrive."
    )


TRAIN_PATH = locate_path("data")
SFT_ADAPTER_PATH = locate_path(SFT_ADAPTER_DIR)

# Resolve to actual data file
if TRAIN_PATH.is_dir():
    TRAIN_JSON = TRAIN_PATH / "train.json"
else:
    TRAIN_JSON = TRAIN_PATH

if not TRAIN_JSON.is_file():
    raise FileNotFoundError(f"train.json not found at {TRAIN_JSON}")

ADAPTER_FILE = SFT_ADAPTER_PATH / "adapter_model.safetensors"
if ADAPTER_FILE.is_file():
    print("SFT adapter found:", ADAPTER_FILE)
else:
    print("WARNING: SFT adapter not found — GRPO will start from base model weights.")
    ADAPTER_FILE = None

print("Training data:", TRAIN_JSON)

GPU: Tesla T4
BF16 supported: False
SFT adapter found: /content/telelogs/qwen35_4b_sft_lora/adapter_model.safetensors
Training data: /content/telelogs/data/train.json


## 3. Load model and attach SFT adapter

`fast_inference=True` enables vLLM for GRPO generation. The LoRA rank matches the SFT adapter (r=16, alpha=16) so the loaded weights are compatible.

In [7]:
import torch

# A100 Optimization: Enable TF32 for faster matrix multiplications
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    fast_inference=False,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Ensure Qwen3.5 chat template defaults to enable_thinking=True
# so GRPOTrainer's apply_chat_template calls produce <think> blocks.
_original_apply = tokenizer.apply_chat_template


def _apply_with_thinking(*args, **kwargs):
    kwargs.setdefault("enable_thinking", True)
    return _original_apply(*args, **kwargs)


tokenizer.apply_chat_template = _apply_with_thinking

print("Base model loaded:", MODEL_NAME)

# --- load SFT adapter weights into the PEFT model ---
if ADAPTER_FILE is not None:
    from safetensors.torch import load_file as safe_load

    adapter_weights = safe_load(str(ADAPTER_FILE))
    model_state = model.state_dict()

    def normalize_key(k):
        while True:
            changed = False
            for prefix in ["base_model.", "model.", "language_model."]:
                if k.startswith(prefix):
                    k = k[len(prefix):]
                    changed = True
            if not changed: break
        return k.replace(".default.", ".")
        
    adapter_norm = {normalize_key(k): v for k, v in adapter_weights.items()}

    loaded = 0
    for key, param in model_state.items():
        if "lora_" not in key:
            continue
        norm_key = normalize_key(key)
        if norm_key in adapter_norm:
            param.data.copy_(adapter_norm[norm_key].to(param.device, param.dtype))
            loaded += 1
    total_lora = sum(1 for k in model_state if "lora_" in k)
    print(f"Loaded {loaded}/{total_lora} LoRA parameters from SFT adapter")
    if loaded == 0:
        raise RuntimeError(
            "Failed to load any SFT LoRA weights. Key prefix mismatch — "
            "check that SFT_ADAPTER_DIR contains a valid Unsloth LoRA adapter."
        )
else:
    print("Starting GRPO from base model (no SFT adapter loaded).")

==((====))==  Unsloth 2026.7.5: Fast Qwen3_5 patching. Transformers: 5.14.1. vLLM: 0.15.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.19 GiB. GPU 0 has a total capacity of 14.56 GiB of which 265.81 MiB is free. Including non-PyTorch memory, this process has 14.30 GiB memory in use. Of the allocated memory 14.10 GiB is allocated by PyTorch, and 53.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 4. Prepare GRPO dataset

GRPO only needs the question (prompt) and the ground-truth label (answer). The model will generate its own reasoning at each step.

In [ ]:
def load_grpo_data(path):
    """Load train.json — a single JSON array of {"question": ..., "answer": "C#"}."""
    with open(path, encoding="utf-8") as fh:
        raw = json.load(fh)
    if not isinstance(raw, list):
        raise ValueError("train.json must be a JSON array")
    records = []
    for idx, item in enumerate(raw):
        if not isinstance(item, dict):
            raise ValueError(f"train.json[{idx}]: expected object")
        question = item.get("question")
        answer = item.get("answer")
        if not question or not answer:
            raise ValueError(f"train.json[{idx}]: missing question or answer")
        if not re.fullmatch(r"C[1-8]", answer):
            raise ValueError(
                f"train.json[{idx}]: answer '{answer}' is not a valid C1-C8 label"
            )
        records.append(
            {
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": question},
                ],
                "answer": answer,
            }
        )
    return records


records = load_grpo_data(TRAIN_JSON)
print(f"Loaded {len(records)} GRPO examples")
assert len(records) >= 100, f"Too few examples: {len(records)}"

label_counts = Counter(r["answer"] for r in records)
print("Label distribution:", dict(sorted(label_counts.items())))

# Filter prompts that fit within max_prompt_length
tokenized = [
    len(
        tokenizer.apply_chat_template(
            rec["prompt"], add_generation_prompt=True, tokenize=True
        )
    )
    for rec in records
]
max_prompt_length = int(np.quantile(tokenized, 0.90))
print(f"90th percentile prompt length: {max_prompt_length} tokens")

truncated = [records[i] for i, n in enumerate(tokenized) if n <= max_prompt_length]
print(
    f"Filtered: {len(truncated)}/{len(records)} examples (removed top 10% longest prompts)"
)

max_completion_length = MAX_SEQ_LENGTH - max_prompt_length
print(f"Max completion length: {max_completion_length}")

dataset = Dataset.from_list(truncated)
print("\nFirst prompt (truncated):\n")
first_chat = tokenizer.apply_chat_template(
    dataset[0]["prompt"], add_generation_prompt=True, tokenize=False
)
print(first_chat[:500] + "...")

# Sanity check: prompt starts and has thinking enabled
assert "<|im_start|>assistant\n" in first_chat, "Missing assistant template marker"
assert "<think>" in first_chat, (
    "<think> tag missing from rendered prompt — enable_thinking may be disabled"
)

## 5. Reward functions

Two rewards are combined. GRPO normalises advantages within each generation group, so negative values are fine.

1. **Format reward** (0.0–1.0): explicitly scores the response structure.
2. **Answer reward** (-1.0 / -0.5 / 1.0): extracts the final boxed label and compares to ground truth.

In [ ]:
# Match the terminal boxed C1-C8 label (may have whitespace inside braces)
BOX_RE = re.compile(r"\\boxed\s*\{\s*(C[1-8])\s*\}")

# Match just before EOS or end-of-string
FINAL_BOX_RE = re.compile(r"\\boxed\s*\{\s*(C[1-8])\s*\}\s*$")


def format_reward(completions, **kwargs):
    """Reward well-structured responses: <think>...</think> + terminal boxed answer."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        score = 0.0
        # Opening and closing think tags both present
        if "<think>" in response:
            score += 0.25
        if "</think>" in response:
            score += 0.25
        # Has exactly one boxed C# label? (not counting, just checking presence)
        boxes = BOX_RE.findall(response)
        if boxes:
            score += 0.25
            # Bonus if the LAST box is at the end (well-formed response)
            if FINAL_BOX_RE.search(response):
                score += 0.25
        scores.append(score)
    return scores


def answer_reward(completions, answer, **kwargs):
    """+1.0 for correct boxed label, -0.5 for valid but wrong, -1.0 for invalid."""
    scores = []
    for completion, true_answer in zip(completions, answer):
        response = completion[0]["content"]
        match = FINAL_BOX_RE.search(response)
        if match is None:
            scores.append(-1.0)
        elif match.group(1) == true_answer:
            scores.append(1.0)
        else:
            scores.append(-0.5)
    return scores


# --- quick smoke test ---
test_completions = [
    [{"content": "<think>some reasoning</think>\n\n\\boxed{C3}"}],
    [{"content": "<think>blah</think>\\boxed{C1}"}],
    [{"content": "no tags here \\boxed{C7}"}],
    [{"content": "<think>no close\n\\boxed{C4}"}],
    [{"content": "random text"}],
]
test_answers = ["C3", "C1", "C7", "C4", "C8"]
fmt = format_reward(test_completions)
ans = answer_reward(test_completions, answer=test_answers)
print("Format rewards:", fmt)
print("Answer rewards:", ans)
assert fmt == [1.0, 0.75, 0.75, 0.5, 0.0], f"Unexpected format rewards: {fmt}"
assert ans == [1.0, 1.0, 1.0, 1.0, -1.0], f"Unexpected answer rewards: {ans}"
print("Reward smoke test passed.")

## 6. GRPO training

`GRPOTrainer` generates `num_generations` completions per prompt via vLLM, scores them with the reward functions, and applies a clipped advantage-weighted loss.

**Patience**: expect low or noisy reward for the first 50–100 steps. The reward column should trend upward.

Set `max_steps` to a higher value (e.g. 500) for a full training run.

In [ ]:
from vllm import SamplingParams

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=1.0,
    top_k=-1,
    seed=SEED,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=1.0,
    learning_rate=1e-6,
    beta=0.04,
    epsilon=0.2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # Optimized for A100
    num_generations=8,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=250,
    save_steps=250,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    output_dir=OUTPUT_DIR,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward,
        answer_reward,
    ],
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

## 7. Evaluate a few prompts

Spot-check a handful of training questions with the GRPO-trained model.

In [ ]:
import torch

# A100 Optimization: Enable TF32 for faster matrix multiplications
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import json

FastLanguageModel.for_inference(model)
model.eval()

sample_count = 5
samples = random.sample(truncated, min(sample_count, len(truncated)))

for i, rec in enumerate(samples):
    prompt = tokenizer.apply_chat_template(
        rec["prompt"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(
        model.device
    )
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(
        output_ids[0, prompt_length:], skip_special_tokens=True
    ).strip()
    match = FINAL_BOX_RE.search(completion)
    predicted = match.group(1) if match else "INVALID"
    correct = "✓" if predicted == rec["answer"] else "✗"
    print(
        f"\n--- Sample {i + 1}  Target={rec['answer']}  Predicted={predicted} {correct} ---"
    )
    # Show last 300 chars of completion
    short = completion[-400:] if len(completion) > 400 else completion
    print(short)

## 8. Save the GRPO LoRA adapter

This adapter stacks on top of the SFT adapter. Load it with:

```python
model, tokenizer = FastLanguageModel.from_pretrained("unsloth/Qwen3.5-4B", ...)
model = FastLanguageModel.get_peft_model(model, ...)
# Load SFT adapter first
model.load_lora("qwen35_4b_sft_lora")
# Then load GRPO adapter
model.load_lora("qwen35_4b_grpo_lora")
```

Or merge both adapters before saving.

In [ ]:
import shutil

model.save_pretrained(GRPO_ADAPTER_DIR)
tokenizer.save_pretrained(GRPO_ADAPTER_DIR)

with open(
    Path(GRPO_ADAPTER_DIR) / "training_handoff.json", "w", encoding="utf-8"
) as fh:
    json.dump(
        {
            "base_model": MODEL_NAME,
            "sft_adapter": SFT_ADAPTER_DIR,
            "max_seq_length": MAX_SEQ_LENGTH,
            "lora_rank": LORA_RANK,
            "fast_inference": True,
            "response_contract": "<think>...</think>\\n\\n\\boxed{identity}",
            "grpo_note": (
                "This LoRA adapter was trained via GRPO on top of the SFT adapter. "
                "Load the SFT adapter FIRST, then stack this adapter. "
                "Both must use the same Qwen3.5 base and tokenizer."
            ),
        },
        fh,
        indent=2,
    )

adapter_archive = shutil.make_archive(
    GRPO_ADAPTER_DIR, "zip", root_dir=GRPO_ADAPTER_DIR
)
print("GRPO adapter archive:", adapter_archive)

try:
    from google.colab import files

    files.download(adapter_archive)
except ImportError:
    print("Not running in Colab; archive remains in the current directory.")